# TikTok Day-30 Views Forecast
WeCloudData DS Bootcamp, Week 6 competition (Kaggle: `predictive-modelling-ds`)

**Goal:** predict `target_day30_views` for the 3,001 videos in `test_videos.csv` using only days 0-5. **Metric:** RMSE.

**Data:** the full Hugging Face dataset (`lingbow/tiktok-video-engagement-200k`). 70,939 labelled videos are used for training (the competition test videos are excluded).

**Steps**
1. Setup
2. Load data and prevent leakage
3. Feature engineering (days 0-5 only)
4. Model
5. Validation
6. Train on all labelled videos and write the submission
7. Checks

## 1. Setup

In [1]:
import os, time, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import lightgbm as lgb
from sklearn.model_selection import KFold

SEED = 42
np.random.seed(SEED)


def find_dir(filename):
    cands = [os.environ.get('DATA_DIR', ''), os.environ.get('HF_DIR', ''), '.', './data', './hf', '/kaggle/input/predictive-modelling-ds', '/content']
    return next((d for d in cands if d and os.path.exists(os.path.join(d, filename))), None)


KAGGLE_DIR = find_dir('test_videos.csv')          # competition files: test_videos.csv, sample_submission.csv
HF_DIR = find_dir('engagement_daily.parquet')     # Hugging Face files: videos.parquet, engagement_daily.parquet, creator_daily.parquet
assert KAGGLE_DIR, 'test_videos.csv / sample_submission.csv not found (set DATA_DIR)'
assert HF_DIR, 'videos.parquet, engagement_daily.parquet, creator_daily.parquet not found (set HF_DIR)'
print('competition files:', KAGGLE_DIR, '| Hugging Face files:', HF_DIR)

competition files: ./data | Hugging Face files: ./hf


## 2. Load data and prevent leakage
* Only numeric and categorical columns are read from `videos.parquet`. Text columns are never loaded.
* The day-30 view count is stored as the label; every engagement row after day 5 is dropped before any feature is built.
* The 3,001 test videos are removed from the training pool.

In [2]:
VCOLS = ['video_id', 'author_id', 'create_time', 'create_date', 'duration', 'ratio', 'desc_language', 'is_english', 'created_by_ai', 'is_ads',
         'music_selected_from', 'music_author', 'music_owner_id', 'music_id', 'word_count', 'emoji_count', 'question_count', 'hashtag_count',
         'speaking_rate', 'topic', 'anger', 'joy', 'surprise', 'sadness', 'disgust', 'fear']
videos = pd.read_parquet(os.path.join(HF_DIR, 'videos.parquet'), columns=VCOLS)
for c in ['is_ads', 'music_owner_id', 'music_id']:
    videos[c] = pd.to_numeric(videos[c])
eng_full = pd.read_parquet(os.path.join(HF_DIR, 'engagement_daily.parquet'))
creat = pd.read_parquet(os.path.join(HF_DIR, 'creator_daily.parquet'))
test_ids = set(pd.read_csv(os.path.join(KAGGLE_DIR, 'test_videos.csv'), dtype={'video_id': str}, usecols=['video_id']).video_id)
sample_sub = pd.read_csv(os.path.join(KAGGLE_DIR, 'sample_submission.csv'), dtype={'video_id': str})

labels = eng_full.loc[eng_full.days_since_post == 30].set_index('video_id')['play_count'].astype(float)   # label: views on day 30
eng = eng_full.loc[eng_full.days_since_post <= 5].copy()                                                  # LEAKAGE GUARD: days 0-5 only
del eng_full
assert eng.days_since_post.max() <= 5

pool_ids = [i for i in labels.index if i not in test_ids]      # training pool: labelled videos without the test videos
assert not (set(pool_ids) & test_ids) and test_ids <= set(videos.video_id)
last = eng.sort_values(['video_id', 'days_since_post']).groupby('video_id').tail(1).set_index('video_id')['play_count']
P_pool = last.reindex(pool_ids).fillna(0).values; y_pool = labels.loc[pool_ids].values
print('videos', len(videos), '| labelled', len(labels), '| training pool', len(pool_ids), '| test', len(test_ids))

videos 209543 | labelled 73940 | training pool 70939 | test 3001


## 3. Feature engineering (days 0-5 only)
* Growth speed over the last 1, 2 and 3 days, log growth, acceleration, decay of the daily increments.
* Like, comment, share, save and download rates.
* Author and music context from the author's other videos (days 0-5 features only, no labels).
* Creator snapshot (cut at post date + 5 days), video metadata, resolution, original-sound flag.

In [3]:
MET = ['play_count', 'like_count', 'comment_count', 'share_count', 'collect_count', 'download_count', 'whatsapp_share_count']
SHORT = dict(zip(MET, ['p', 'l', 'c', 's', 'k', 'd', 'w']))


def engagement_features(vids, en):
    en = en.sort_values(['video_id', 'days_since_post']).copy()
    for m in MET:                                            # counters are cumulative -> enforce monotonicity (tiny noise)
        en[m] = en.groupby('video_id')[m].cummax()
    wide = {}
    for m in MET:
        pv = en.pivot(index='video_id', columns='days_since_post', values=m).reindex(columns=range(6))
        for d in range(6):
            wide[f'{SHORT[m]}{d}'] = pv[d]
    W = pd.DataFrame(wide).reindex(vids)
    last_day = en.groupby('video_id').days_since_post.max().reindex(vids)
    last = en.groupby('video_id').tail(1).set_index('video_id').reindex(vids)
    F = pd.DataFrame(index=vids)
    F['last_day'] = last_day
    F['n_days'] = en.groupby('video_id').size().reindex(vids)
    for m in MET:
        F[f'{SHORT[m]}_last'] = last[m]
    P = F['p_last']
    F['logP'] = np.log1p(P)
    for m in MET[1:]:
        F[f'{SHORT[m]}_rate'] = F[f'{SHORT[m]}_last'] / (P + 1)
    F['c_per_l'] = F['c_last'] / (F['l_last'] + 1); F['s_per_l'] = F['s_last'] / (F['l_last'] + 1)
    F['k_per_l'] = F['k_last'] / (F['l_last'] + 1); F['s_per_c'] = F['s_last'] / (F['c_last'] + 1)
    for d in range(6):
        F[f'p{d}'] = W[f'p{d}']; F[f'pf{d}'] = W[f'p{d}'] / (P + 1)
    for d in range(1, 6):
        inc = W[f'p{d}'] - W[f'p{d-1}']; F[f'inc{d}'] = inc; F[f'incf{d}'] = inc / (P + 1)

    def back(col, k):                                        # value k days before the LAST observed day
        out = pd.Series(np.nan, index=vids)
        for d in range(6):
            mask = (last_day - k) == d
            out[mask] = W.loc[mask, f'{col}{d}']
        return out
    for k in (1, 2, 3):
        pk, lk, sk = back('p', k), back('l', k), back('s', k)
        F[f'p_back{k}'] = pk
        F[f'vel{k}'] = (P - pk) / (P + 1)                    # share of plays gained in the last k days
        F[f'grow{k}'] = np.log1p(P) - np.log1p(pk)
        F[f'lvel{k}'] = (F['l_last'] - lk) / (F['l_last'] + 1)
        F[f'l_inc_rate{k}'] = (F['l_last'] - lk) / (P - pk + 1)   # likes per *new* play recently
        F[f'svel{k}'] = (F['s_last'] - sk) / (F['s_last'] + 1)
    F['accel'] = F['vel1'] / (F['vel2'] - F['vel1'] + 1e-3)
    F['inc_ratio'] = (P - back('p', 1) + 1) / (back('p', 1) - back('p', 2) + 1)
    F['p5_over_p1'] = np.log1p(P) - np.log1p(W['p1'])
    F['first_day_plays'] = W[['p0', 'p1']].bfill(axis=1)['p0']
    F['first_over_last'] = F['first_day_plays'] / (P + 1)
    F['l_rate_d1'] = W['l1'] / (W['p1'] + 1)
    F['l_rate_change'] = F['l_rate'] - F['l_rate_d1']
    return F


def creator_features(vdf, cr):
    # STRICT cutoff: only snapshots dated <= create_date + 5 days (and one taken before posting)
    cr = cr.copy(); cr['date'] = pd.to_datetime(cr['date']); cr = cr.drop(columns=['enterprise_verified'])
    v = vdf[['video_id', 'author_id', 'create_date']].copy(); v['create_date'] = pd.to_datetime(v['create_date'])
    v['cutoff'] = v['create_date'] + pd.Timedelta(days=5); v['pre_cut'] = v['create_date'] - pd.Timedelta(days=1)
    cols = ['follower_count', 'following_count', 'total_favorited', 'video_count']

    def snap(cutcol, name):
        m = pd.merge_asof(v[['video_id', 'author_id', cutcol]].sort_values(cutcol), cr.sort_values('date'),
                          left_on=cutcol, right_on='date', by='author_id', direction='backward').set_index('video_id')[cols]
        m.columns = [f'{name}_{c}' for c in cols]
        return m
    C = snap('cutoff', 'post5').join(snap('pre_cut', 'pre')).reindex(v['video_id'])
    for c in cols:
        C[f'd_{c}'] = C[f'post5_{c}'] - C[f'pre_{c}']
    C['fav_per_follower'] = C['post5_total_favorited'] / (C['post5_follower_count'] + 1)
    return C


def meta_features(df):
    F = pd.DataFrame(index=df['video_id'])
    ct = pd.to_datetime(df['create_time'].values)
    F['hour'] = ct.hour; F['dow'] = ct.dayofweek
    for c in ['duration', 'is_english', 'created_by_ai', 'is_ads', 'word_count', 'emoji_count', 'question_count', 'hashtag_count',
              'speaking_rate', 'anger', 'joy', 'surprise', 'sadness', 'disgust', 'fear']:
        F[c] = df[c].values
    F['res'] = df['ratio'].str.replace('p', '', regex=False).astype(float).values     # 540p / 720p ...
    F['music_original'] = (df['music_selected_from'].astype(str) == 'original').astype(float).values
    return F


def context_features(allv, E):
    # unsupervised context from other videos of the same author / same music (days 0-5 features only, no labels)
    ctx = pd.DataFrame({'author_id': allv['author_id'].values, 'music_id': allv['music_id'].values,
                        'create_date': pd.to_datetime(allv['create_date'].values)}, index=allv['video_id'])
    for c in ['logP', 'vel2', 'l_rate']:
        ctx[c] = E.loc[ctx.index, c].values
    g = ctx.groupby('author_id'); n = g['logP'].transform('size'); den = (n - 1).replace(0, np.nan)
    out = pd.DataFrame(index=ctx.index)
    out['a_nvid'] = n
    out['a_mean_logP_loo'] = (g['logP'].transform('sum') - ctx['logP']) / den          # mean of the author's OTHER videos
    out['a_mean_logP'] = g['logP'].transform('mean'); out['a_max_logP'] = g['logP'].transform('max'); out['a_std_logP'] = g['logP'].transform('std')
    out['logP_minus_author'] = ctx['logP'] - out['a_mean_logP_loo']
    out['logP_rank_in_author'] = g['logP'].rank(pct=True)
    out['a_mean_vel2_loo'] = (g['vel2'].transform('sum') - ctx['vel2'].fillna(0)) / den
    out['a_mean_lrate'] = g['l_rate'].transform('mean')
    out['music_cnt'] = ctx.groupby('music_id')['logP'].transform('size'); out['music_mean_logP'] = ctx.groupby('music_id')['logP'].transform('mean')
    out['a_first_date_gap'] = (ctx['create_date'] - g['create_date'].transform('min')).dt.days
    return out


def build_features(train, test, eng, creat):
    allv = pd.concat([train.drop(columns=['target_day30_views']), test], ignore_index=True)
    ids = allv['video_id'].values
    E = engagement_features(ids, eng)
    C = creator_features(allv, creat).reindex(ids)
    M = meta_features(allv)
    ctx = context_features(allv, E)
    X = pd.concat([E, C, M, ctx], axis=1)
    X['plays_per_follower'] = X['p_last'] / (X['post5_follower_count'] + 1)
    X['logP_minus_logF'] = X['logP'] - np.log1p(X['post5_follower_count'])
    X['plays_per_follower_gain'] = X['p_last'] / (X['d_follower_count'].clip(lower=0) + 1)
    return X

In [4]:
t0 = time.time()
ids = videos['video_id'].values
E = engagement_features(ids, eng)
C = creator_features(videos, creat).reindex(ids)
M = meta_features(videos)
ctx_f = context_features(videos, E)
X_all = pd.concat([E, C, M, ctx_f], axis=1)
X_all['plays_per_follower'] = X_all['p_last'] / (X_all['post5_follower_count'] + 1)
X_all['logP_minus_logF'] = X_all['logP'] - np.log1p(X_all['post5_follower_count'])
X_all['plays_per_follower_gain'] = X_all['p_last'] / (X_all['d_follower_count'].clip(lower=0) + 1)
X_all.index = ids
X_all = X_all.copy()

XT = X_all.loc[pool_ids]                                  # training pool
XTEST = X_all.loc[sample_sub.video_id]                    # test videos, in submission order
assert not (set(XT.index) & set(XTEST.index))

allc = list(X_all.columns)
dyn = list(dict.fromkeys([c for c in allc if c[:3] in ('vel', 'gro', 'lve', 'svel', 'l_i')] + [
    'accel', 'inc_ratio', 'p5_over_p1', 'first_over_last', 'l_rate_d1', 'l_rate_change', 'logP', 'p_last', 'l_rate', 'c_rate',
    's_rate', 'k_rate', 'd_rate', 'w_rate', 'c_per_l', 's_per_l', 'k_per_l', 's_per_c', 'last_day', 'n_days']))
ctx = [c for c in allc if c.startswith(('a_', 'music_cnt', 'music_mean', 'logP_minus_author', 'logP_rank'))]
TAIL_COLS = dyn + ctx + ['music_original', 'res']
print(f'feature matrix {X_all.shape} built in {time.time()-t0:.0f}s | {len(TAIL_COLS)} features used by the viral-video model')

feature matrix (209543, 114) built in 12s | 46 features used by the viral-video model


## 4. Model
* **Target:** growth after day 5, `views_day30 / views_last_observed`. Forecasts are floored at the observed views and the growth ratio is clipped to [1, 6].
* **Weights:** each video is weighted by (views on day 5)², so the training loss matches raw-scale RMSE.
* **Videos with more than 5,000 views:** average of two LightGBM models (log growth with a log-normal correction, and capped growth ratio), each averaged over 5 seeds.
* **Other videos:** one small LightGBM.

In [5]:
TAIL_THR = 5000        # videos with more views than this on their last observed day use the "viral" model
RATIO_CAP = 6.0
rmse = lambda a, b: float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))
yT = y_pool.astype(float); PT = P_pool.astype(float)

GBM_BASE = dict(objective='regression', learning_rate=0.03, num_leaves=15, min_child_samples=50, subsample=0.7, subsample_freq=1,
                colsample_bytree=0.5, reg_lambda=20.0, n_estimators=400, verbose=-1, n_jobs=2)


def gbm_log_tail(Xa, ya, Pa, Xb, cols=None, power=2.0, seed=0):
    cols = cols or TAIL_COLS
    lr = np.log(np.maximum(ya, 1) / (Pa + 1)); w = (Pa + 1) ** power; w = w / w.mean()
    m = lgb.LGBMRegressor(**GBM_BASE, random_state=seed).fit(Xa[cols], lr, sample_weight=w)
    sig2 = np.average((lr - m.predict(Xa[cols])) ** 2, weights=w)
    return np.exp(m.predict(Xb[cols]) + sig2 / 2)


def gbm_ratio_tail(Xa, ya, Pa, Xb, cols=None, power=2.0, cap=10.0, seed=0):
    cols = cols or TAIL_COLS
    r = np.minimum(ya / (Pa + 1), cap); w = (Pa + 1) ** power; w = w / w.mean()
    m = lgb.LGBMRegressor(**GBM_BASE, random_state=seed).fit(Xa[cols], r, sample_weight=w)
    return np.maximum(m.predict(Xb[cols]), 1.0)


def final_tail(Xa, ya, Pa, Xb, n=5):
    a = np.mean([gbm_log_tail(Xa, ya, Pa, Xb, seed=s) for s in range(n)], axis=0)
    b = np.mean([gbm_ratio_tail(Xa, ya, Pa, Xb, seed=s) for s in range(n)], axis=0)
    return 0.5 * a + 0.5 * b


SMALL_PARAMS = dict(objective='regression', learning_rate=0.03, num_leaves=15, min_child_samples=40, subsample=0.7, subsample_freq=1,
                    colsample_bytree=0.5, reg_lambda=20, n_estimators=400, verbose=-1, n_jobs=2)


def small_model(Xa, ya, Pa, Xb, seed=0):
    lr = np.log(np.maximum(ya, 1) / (Pa + 1)); m = lgb.LGBMRegressor(**SMALL_PARAMS, random_state=seed).fit(Xa, lr)
    return np.exp(m.predict(Xb) + np.mean((lr - m.predict(Xa)) ** 2) / 2)


def predict_pipeline(Xa, ya, Xb, seed=0, tail_fn=final_tail):
    Pa = Xa['p_last'].fillna(0).values; Pb = Xb['p_last'].fillna(0).values
    out = np.zeros(len(Xb)); ta, tb = Pa > TAIL_THR, Pb > TAIL_THR
    out[tb] = np.clip(tail_fn(Xa[ta], ya[ta], Pa[ta], Xb[tb]), 1, RATIO_CAP) * (Pb[tb] + 1)
    out[~tb] = np.clip(small_model(Xa[~ta], ya[~ta], Pa[~ta], Xb[~tb], seed), 1, RATIO_CAP) * (Pb[~tb] + 1)
    return np.maximum(out, Pb)

## 5. Validation
5-fold cross-validation on the 70,939 labelled videos. The RMSE is compared with a constant-growth baseline (`1.16 x views on day 5`); a ratio below 1 is better than the baseline.

In [6]:
t0 = time.time()
const_pred = 1.16 * PT
oof = np.zeros(len(yT))
for a, b in KFold(5, shuffle=True, random_state=0).split(XT):
    oof[b] = predict_pipeline(XT.iloc[a], yT[a], XT.iloc[b], seed=0)
print('baseline RMSE (1.16 x day-5 views): %.0f' % rmse(const_pred, yT))
print('model CV RMSE                     : %.0f' % rmse(oof, yT))
print('ratio to baseline                 : %.3f  (%.0fs)' % (rmse(oof, yT) / rmse(const_pred, yT), time.time() - t0))

baseline RMSE (1.16 x day-5 views): 147738
model CV RMSE                     : 123250
ratio to baseline                 : 0.834  (59s)


## 6. Train on all labelled videos and write the submission

In [7]:
t0 = time.time()
test_pred = predict_pipeline(XT, yT, XTEST, seed=SEED)
sub = pd.DataFrame({'video_id': sample_sub.video_id.values, 'target_day30_views': np.round(test_pred).astype(np.int64)})
sub.to_csv('submission_full_data.csv', index=False)
print(f'trained on {len(XT)} videos, predicted {len(sub)} test videos in {time.time()-t0:.0f}s')
sub.head()

trained on 70939 videos, predicted 3001 test videos in 13s


,video_id,target_day30_views
0,7400582591771938090,254
1,7403167206978374958,388
2,7435124823820356906,2227
3,7409800970143714606,296
4,7410935177553333550,266


## 7. Checks

In [8]:
P_test = XTEST['p_last'].fillna(0).values
assert list(sub.columns) == ['video_id', 'target_day30_views']
assert len(sub) == len(sample_sub) and sub.video_id.is_unique and (sub.video_id.values == sample_sub.video_id.values).all()
assert sub.target_day30_views.notna().all() and (sub.target_day30_views >= 0).all()
assert (sub.target_day30_views.values >= np.round(P_test)).all(), 'forecast below the views already observed'
assert not (set(XT.index) & test_ids), 'a test video is in the training data'
assert eng.days_since_post.max() <= 5, 'engagement after day 5 is present'
print('all checks passed')

all checks passed


| Rule | How it is met |
|---|---|
| Tabular data only | text columns are never loaded; no language models or embeddings |
| No test information | test video ids are excluded from training and their day-30 values are never used |
| Days 0-5 only | engagement after day 5 is dropped before features are built; creator snapshots are cut at post date + 5 days; context features use other videos' features only, never labels |
| No AutoML | hand-built features, LightGBM and scikit-learn |
| Solo and reproducible | fixed seeds; the notebook regenerates the submission from the raw files |